In [ ]:
import os
import json
from datasets import load_dataset

from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas import evaluate

from core.embedding.sentence_transformer_embedder import LocalEmbedder
from openai import OpenAI
from datasets import load_dataset
from core.rag import default_rag_client
from dotenv import load_dotenv

load_dotenv()

In [ ]:
dataset = load_dataset("galileo-ai/ragbench", "techqa", split="train")
sampled_data = dataset.select(range(min(1000, len(dataset))))

seen_docs = set()

os.makedirs("./rag_staging_files", exist_ok=True)
for row in sampled_data:
    for doc in row["documents"]:
        doc_hash = hash(doc)
        if doc_hash not in seen_docs:
            seen_docs.add(doc_hash)
            file_path = f"../rag_staging_files/doc_{doc_hash}.txt"
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(doc)


In [ ]:
all_evaluation_samples = []

# Assuming you uploaded the staging files to your RAG system and extracted 
# a global manifest of all generated chunks: 
# system_chunks = [{"id": "chunk_101", "text": "...", "source": "doc_xyz"}, ...]

for row in dataset:
    sentence_lookup = {}
    for doc_sentences in row["documents_sentences"]:
        for sent_id, sent_text in doc_sentences:
            sentence_lookup[sent_id] = sent_text.strip()
            
    target_sentences = [
        sentence_lookup[key] 
        for key in row["all_relevant_sentence_keys"] 
        if key in sentence_lookup
    ]
    
    reference_context_ids = []
    for chunk in system_chunks:
        for target_sent in target_sentences:
            if target_sent in chunk["text"]:
                reference_context_ids.append(chunk["id"])
                break
                
    retrieved_chunks = my_rag.query(row["question"], top_k=5)
    retrieved_context_ids = [chunk["id"] for chunk in retrieved_chunks]
    
    sample = SingleTurnSample(
        retrieved_context_ids=retrieved_context_ids,
        reference_context_ids=list(set(reference_context_ids))
    )
    all_evaluation_samples.append(sample)

In [ ]:
import pandas as pd
from ragas.metrics import IDBasedContextRecall, IDBasedContextPrecision

id_recall = IDBasedContextRecall()
id_precision = IDBasedContextPrecision()

results = []

for sample in all_evaluation_samples:
    if not sample.reference_context_ids:
        continue
        
    recall_score = await id_recall.single_turn_ascore(sample)
    precision_score = await id_precision.single_turn_ascore(sample)
    
    results.append({
        "retrieved_ids": sample.retrieved_context_ids,
        "reference_ids": sample.reference_context_ids,
        "recall": recall_score,
        "precision": precision_score
    })

df_results = pd.DataFrame(results)
print(f"Mean Recall: {df_results['recall'].mean():.4f}")
print(f"Mean Precision: {df_results['precision'].mean():.4f}")
display(df_results.head())